In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

base = Path(r"Areas-of-interest-POIs\Data-created(2nd LLM run)")

file_paths = [
    base / "final_predictions_with_geometry_rerun.gpkg",
    base / "final_predictions_with_geometry_rerun-0.gpkg",
    base / "final_predictions_with_geometry_rerun-1.gpkg",
    base / "final_predictions_with_geometry_rerun-2.gpkg",
    base / "final_predictions_with_geometry_rerun-3.gpkg",
    base / "final_predictions_with_geometry_rerun-4.gpkg",
    base / "final_predictions_with_geometry_rerun-5.gpkg",
    base / "final_predictions_with_geometry_rerun-6.gpkg",
    base / "final_predictions_with_geometry_rerun-7.gpkg",
    base / "final_predictions_with_geometry_rerun-8.gpkg",
    base / "final_predictions_with_geometry_rerun-9.gpkg",

]

merged_gdf = None
stats = []

for path in file_paths:
    if not path.exists():
        print(f"File not found: {path.name}")
        continue

    try:
        new_gdf = gpd.read_file(path)

        if len(new_gdf) == 0:
            print(f"Skipped empty file: {path.name}")
            continue

        new_gdf["gml_id"] = new_gdf["gml_id"].astype(str)

        # remove duplicates inside the same file
        internal_duplicates = new_gdf["gml_id"].duplicated().sum()
        new_gdf = new_gdf.drop_duplicates(subset="gml_id", keep="last")

        if merged_gdf is None:
            merged_gdf = new_gdf.copy()

            stats.append({
                "file": path.name,
                "input_rows": len(new_gdf),
                "internal_duplicates_removed": internal_duplicates,
                "added": len(new_gdf),
                "replaced": 0,
                "final_rows_after_file": len(merged_gdf),
            })

            print(f"Loaded base {path.name}: {len(new_gdf)} rows")
            continue

        existing_ids = set(merged_gdf["gml_id"])
        new_ids = set(new_gdf["gml_id"])

        replaced_ids = existing_ids & new_ids
        added_ids = new_ids - existing_ids

        # remove old rows with same gml_id
        merged_gdf = merged_gdf[~merged_gdf["gml_id"].isin(replaced_ids)]

        # add new rows
        merged_gdf = pd.concat([merged_gdf, new_gdf], ignore_index=True)

        # convert back to GeoDataFrame
        merged_gdf = gpd.GeoDataFrame(
            merged_gdf,
            geometry="geometry",
            crs=new_gdf.crs
        )

        stats.append({
            "file": path.name,
            "input_rows": len(new_gdf),
            "internal_duplicates_removed": internal_duplicates,
            "added": len(added_ids),
            "replaced": len(replaced_ids),
            "final_rows_after_file": len(merged_gdf),
        })

        print(
            f"Processed {path.name}: "
            f"added={len(added_ids)}, replaced={len(replaced_ids)}, "
            f"final={len(merged_gdf)}"
        )

    except Exception as e:
        print(f"Error loading {path.name}: {e}")


stats_df = pd.DataFrame(stats)

print("\nMerge statistics:")
print(stats_df)

print("\nFinal summary:")
print("Final merged rows:", len(merged_gdf))
print("Duplicate gml_id after merge:", merged_gdf["gml_id"].duplicated().sum())
print("Unique gml_id:", merged_gdf["gml_id"].nunique())

Loaded base final_predictions_with_geometry_rerun.gpkg: 561338 rows
Processed final_predictions_with_geometry_rerun-0.gpkg: added=0, replaced=27042, final=561338
Processed final_predictions_with_geometry_rerun-1.gpkg: added=13566, replaced=0, final=574904
Processed final_predictions_with_geometry_rerun-2.gpkg: added=2476, replaced=0, final=577380
Processed final_predictions_with_geometry_rerun-3.gpkg: added=700, replaced=0, final=578080
Processed final_predictions_with_geometry_rerun-4.gpkg: added=0, replaced=9891, final=578080
Processed final_predictions_with_geometry_rerun-5.gpkg: added=0, replaced=2228, final=578080
Processed final_predictions_with_geometry_rerun-6.gpkg: added=0, replaced=3400, final=578080
Processed final_predictions_with_geometry_rerun-7.gpkg: added=0, replaced=1032, final=578080
Processed final_predictions_with_geometry_rerun-8.gpkg: added=0, replaced=712, final=578080
Processed final_predictions_with_geometry_rerun-9.gpkg: added=0, replaced=68, final=578080

Mer

In [2]:
df = merged_gdf

df.columns

Index(['index', 'gml_id', 'volume_m3', 'label_en', 'osm_building_type',
       'osm_landuse_class', 'osm_landuse_name', 'gfk_class',
       'ALKIS_Landuse_info', 'osm_names', 'alkis_address', 'email', 'amenity',
       'building', 'shop', 'tourism', 'information', 'tags_search',
       'additional_information', 'website', 'sentence', 'interpreted_type',
       'mid_labels', 'bosserhof_class', 'error', 'mid_label', 'geometry',
       'gml_id_pred', 'short_reason', 'level_0'],
      dtype='object')

In [3]:
df.head()

,index,gml_id,volume_m3,label_en,osm_building_type,osm_landuse_class,osm_landuse_name,gfk_class,ALKIS_Landuse_info,osm_names,...,sentence,interpreted_type,mid_labels,bosserhof_class,error,mid_label,geometry,gml_id_pred,short_reason,level_0
0,4.0,13,89.209230,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,general_building_context: building_label=Build...,office building,['work'],normal office,None,['work'],MULTIPOLYGON Z (((608781.076 5797034.372 86.80...,NaN,NaN,NaN
1,5.0,14,119.566130,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,general_building_context: building_label=Build...,office building,['work'],normal office,None,['work'],"MULTIPOLYGON Z (((609122.06 5797462.983 85.46,...",NaN,NaN,NaN
2,6.0,15,1033.650498,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,general_building_context: building_label=Build...,office building,['work'],normal office,None,['work'],MULTIPOLYGON Z (((609428.62 5797079.031 79.962...,NaN,NaN,NaN
3,7.0,16,46.560366,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,general_building_context: building_label=Build...,office building,['work'],normal office,None,['work'],"MULTIPOLYGON Z (((609661.688 5797071.969 77.4,...",NaN,NaN,NaN
4,8.0,0,931.423451,Buildings for business or commerce,None,residential,None,Gebäude,commercial,None,...,general_building_context: building_label=Build...,office building,['work'],normal office,None,['work'],MULTIPOLYGON Z (((609554.181 5797264.172 78.94...,NaN,NaN,NaN


In [4]:
df = df.drop(columns=['sentence', 'interpreted_type','mid_labels', 'error','gml_id_pred', 'short_reason', 'level_0'],axis=1)
df.head()

,index,gml_id,volume_m3,label_en,osm_building_type,osm_landuse_class,osm_landuse_name,gfk_class,ALKIS_Landuse_info,osm_names,...,building,shop,tourism,information,tags_search,additional_information,website,bosserhof_class,mid_label,geometry
0,4.0,13,89.209230,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,None,None,None,None,None,None,None,normal office,['work'],MULTIPOLYGON Z (((608781.076 5797034.372 86.80...
1,5.0,14,119.566130,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,None,None,None,None,None,None,None,normal office,['work'],"MULTIPOLYGON Z (((609122.06 5797462.983 85.46,..."
2,6.0,15,1033.650498,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,None,None,None,None,None,None,None,normal office,['work'],MULTIPOLYGON Z (((609428.62 5797079.031 79.962...
3,7.0,16,46.560366,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,None,None,None,None,None,None,None,normal office,['work'],"MULTIPOLYGON Z (((609661.688 5797071.969 77.4,..."
4,8.0,0,931.423451,Buildings for business or commerce,None,residential,None,Gebäude,commercial,None,...,None,None,None,None,None,None,None,normal office,['work'],MULTIPOLYGON Z (((609554.181 5797264.172 78.94...


In [5]:
df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 578080 entries, 0 to 578079
Data columns (total 23 columns):
 #   Column                  Non-Null Count   Dtype   
---  ------                  --------------   -----   
 0   index                   569875 non-null  float64 
 1   gml_id                  578080 non-null  object  
 2   volume_m3               578080 non-null  float64 
 3   label_en                533577 non-null  object  
 4   osm_building_type       51670 non-null   object  
 5   osm_landuse_class       519989 non-null  object  
 6   osm_landuse_name        47278 non-null   object  
 7   gfk_class               533577 non-null  object  
 8   ALKIS_Landuse_info      509273 non-null  object  
 9   osm_names               22799 non-null   object  
 10  alkis_address           542662 non-null  object  
 11  email                   909 non-null     object  
 12  amenity                 7980 non-null    object  
 13  building                5145 non-null    object  
 

In [6]:
df_final = df[['gml_id','volume_m3','osm_names','bosserhof_class','mid_label','geometry']]
df_final.head()

,gml_id,volume_m3,osm_names,bosserhof_class,mid_label,geometry
0,13,89.209230,None,normal office,['work'],MULTIPOLYGON Z (((608781.076 5797034.372 86.80...
1,14,119.566130,None,normal office,['work'],"MULTIPOLYGON Z (((609122.06 5797462.983 85.46,..."
2,15,1033.650498,None,normal office,['work'],MULTIPOLYGON Z (((609428.62 5797079.031 79.962...
3,16,46.560366,None,normal office,['work'],"MULTIPOLYGON Z (((609661.688 5797071.969 77.4,..."
4,0,931.423451,None,normal office,['work'],MULTIPOLYGON Z (((609554.181 5797264.172 78.94...


In [7]:
df['mid_label'].value_counts()

mid_label
['work']                                    218113
['work' 'errands']                            8204
['work' 'business']                           6795
['leisure']                                   4713
['errands' 'work']                            2552
                                             ...  
['business' 'leisure']                           1
['leisure' 'business' 'meetup' 'work']           1
['errands' 'leisure' 'meetup']                   1
['work' 'childcare' 'sports']                    1
['leisure' 'errands' 'retail_non_daily']         1
Name: count, Length: 369, dtype: int64

In [8]:
df_final = df_final.dropna(subset=['mid_label'])
df_final.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 263513 entries, 0 to 576280
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   gml_id           263513 non-null  object  
 1   volume_m3        263513 non-null  float64 
 2   osm_names        21393 non-null   object  
 3   bosserhof_class  263513 non-null  object  
 4   mid_label        263513 non-null  object  
 5   geometry         263513 non-null  geometry
dtypes: float64(1), geometry(1), object(4)
memory usage: 14.1+ MB


In [9]:
import re

def clean_bosserhof_label(val):
    if pd.isna(val):
        return val  # keep NaN as is
    
    # convert to string (safety)
    val = str(val)
    
    # 1. lowercase
    val = val.lower()
    
    # 2. remove special characters (keep letters, numbers, spaces)
    val = re.sub(r'[^a-z0-9\s]', ' ', val)
    
    # 3. remove extra spaces
    val = re.sub(r'\s+', ' ', val).strip()
    
    return val

# Apply to column
df_final['bosserhof_class'] = df_final['bosserhof_class'].apply(clean_bosserhof_label)

In [10]:
df_final['bosserhof_class'].value_counts()

bosserhof_class
normal office                                                                                        129566
services                                                                                              67015
industrial operations production                                                                      10230
retail small scale                                                                                    10058
retail                                                                                                 9675
customer oriented services                                                                             7420
facilities for culture leisure and sports                                                              4123
yards depots storage areas construction yards                                                          3185
highly productive industries machine material or space intensive                                       3167
entertainmen

In [11]:
raw_text = """
Transport
Yards, depots, storage areas, construction yards
Industrial operations / Production
highly productive industries / machine / material or space intensive
others
Crafts and trades
craft businesses
craft courtyards
Services
normal office
open-plan office
business-oriented services
customer-oriented services
hotels
hotels with conference areas
restaurants / gastronomy
suppliers for car dealerships
vehicle / electrical repair
customer service
car dealerships
Retail
wholesale
retail (small-scale)
discount stores
DIY stores
furniture stores
hypermarkets / superstores
shopping centers
self-service department stores
department stores
factory outlet centers
Public facilities
schools
universities
research institutes
kindergartens
hospitals
nursing homes
Facilities for culture, leisure and sports
entertainment, culture
large cinemas
musical theatres
large discos, fun / leisure pools
arenas, large events
theme parks
fitness / wellness
"""

def clean_label(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)   # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()   # remove extra spaces
    return text

# split into lines, remove empty ones, clean each line
clean_labels = [
    clean_label(line)
    for line in raw_text.splitlines()
    if line.strip()
]

# # print one below another
# for label in clean_labels:
#     print(label)

In [12]:
# count non-empty lines (labels)
num_labels = len([line for line in raw_text.splitlines() if line.strip()])

print(num_labels)

46


In [13]:
import numpy as np

mapping_dict = {
    "others" : "others industrial",
    "industrial operations production others": "industrial operations production",
    "highly productive industries" : "highly productive industries machine material or space intensive",
    "services normal office": "normal office",
    "industrial operations others": "industrial operations production",
    "industrial operations production highly productive industries machine material or space intensive": "highly productive industries machine material or space intensive",
    "yards depots storage areas" : "yards depots storage areas construction yards",
    "services customer oriented services": "customer oriented services",
    "facilities for culture leisure and sports fitness wellness": "fitness wellness",
    "fun leisure pools": "large discos fun leisure pools"
}

parsed_mapping = {}

for k, v in mapping_dict.items():
    left_side = k.split(" : ")[0].strip()
    parsed_mapping[left_side] = v

col = "bosserhof_class"

# keep original just in case
df_final[col + "_original"] = df_final[col]

# apply mapping only where match exists
df_final[col] = df_final[col].replace(parsed_mapping)

# convert None to real NaN
df_final[col] = df_final[col].where(df_final[col].notna(), np.nan)

column_values = set(df_final[col + "_original"].dropna().astype(str).str.strip().unique())
mapping_keys = set(parsed_mapping.keys())

unused_mapping_keys = sorted(mapping_keys - column_values)

if unused_mapping_keys:
    print("These mapping keys were NOT found in the dataframe column:")
    for x in unused_mapping_keys:
        print("-", x)
else:
    print("All mapping keys were found in the dataframe.")

mapped_values = set(parsed_mapping.keys())
remaining_unmapped = sorted(
    set(df_final[col + "_original"].dropna().astype(str).str.strip().unique()) - mapped_values
)

print("\nValues still present in original column that were NOT in your mapping:")
for x in remaining_unmapped:
    print("-", x)

All mapping keys were found in the dataframe.

Values still present in original column that were NOT in your mapping:
- arenas large events
- business oriented services
- car dealerships
- craft businesses
- craft courtyards
- customer oriented services
- customer service
- department stores
- discount stores
- diy stores
- entertainment culture
- facilities for culture leisure and sports
- factory outlet centers
- fitness wellness
- furniture stores
- highly productive industries machine material or space intensive
- hospitals
- hotels
- hotels with conference areas
- hypermarkets superstores
- industrial operations production
- kindergartens
- large cinemas
- large discos fun leisure pools
- normal office
- nursing homes
- open plan office
- public facilities
- research institutes
- restaurants gastronomy
- retail
- retail small scale
- retail wholesale
- schools
- self service department stores
- services
- shopping centers
- suppliers for car dealerships
- theme parks
- transport
-

In [ ]:
### ADD conversion labels of unmatched labels into the dictionary above till all of them are mapped!

In [14]:
len(df_final['bosserhof_class'].unique())

45

In [15]:
df_final['bosserhof_class'].unique()

array(['normal office', 'services',
       'yards depots storage areas construction yards',
       'craft businesses', 'facilities for culture leisure and sports',
       'kindergartens', 'retail', 'industrial operations production',
       'retail small scale', 'fitness wellness', 'entertainment culture',
       'customer oriented services', 'others industrial', 'schools',
       'public facilities', 'restaurants gastronomy', 'diy stores',
       'business oriented services', 'customer service',
       'research institutes',
       'highly productive industries machine material or space intensive',
       'transport', 'discount stores', 'nursing homes',
       'vehicle electrical repair', 'wholesale', 'universities',
       'large discos fun leisure pools', 'furniture stores',
       'car dealerships', 'hotels', 'shopping centers', 'hospitals',
       'arenas large events', 'open plan office',
       'hypermarkets superstores', 'large cinemas', 'department stores',
       'craft court

In [16]:
df_final.head()
df_final = df_final.drop(columns=['bosserhof_class_original'],axis=1)

In [17]:
df_final.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 263513 entries, 0 to 576280
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   gml_id           263513 non-null  object  
 1   volume_m3        263513 non-null  float64 
 2   osm_names        21393 non-null   object  
 3   bosserhof_class  263513 non-null  object  
 4   mid_label        263513 non-null  object  
 5   geometry         263513 non-null  geometry
dtypes: float64(1), geometry(1), object(4)
memory usage: 14.1+ MB


In [18]:
text = """
transport : 200 / 0.5
yards depots storage areas construction yards : 100-150 / 0.7-1,0
industrial operations production 
highly productive industries machine material or space intensive : 100-150 / 0.7-1.0
others industrial : 50-60 / 1.7-2.0
crafts and trades
craft businesses : 40-80 / 1.3-2.5
craft courtyards : 50-60 / 1.7-2.0
services
normal Office : 30-40 / 2.5-3.3
open plan Office :20-30 / 3.3-5.0
business oriented services : 10-50 / 2.0-5.0
customer oriented services : 25-50 / 2.0-4.0
Hotels : 50-100 / 1.0-2.0
hotels with conference Areas : 110-150 / 0.7-0.9 
restaurants gastronomy : 40-80 / 1.3-2.5
suppliers for car dealerships : 60 / 1.7
vehicle electrical repair :30-60 / 1.7-2.3
customer Service : 30 / 3.3
car dealerships : 150 / 0.7
retail
wholesale : 35-50 / 2.0-2.9
retail small scale : 20-50 / 2.5-5.0
discount stores : 90-140 / 0.7-1.1
diy stores : 125-150 / 0.7-0.8
furniture stores: 140-260 / 0.4-0.7
hypermarkets superstores : 70-100 / 1.0-1.4
shopping centers : 25-45 / 2.2-4.0
self service department stores : 85-100 / 1.0-1.2
department stores : 55-75 / 1.3-1.8
factory outlet centers : 40-55 / 1.8-2.5
public facilities
schools : 65-200 / 0.5-1.5
universities : 100-200 / 0.5-1.0
research institutes : 65-100 / 1.0-1.5
kindergartens : 35-60 / 1.7-2.9
hospitals : 65-100 / 1.0-1.5
nursing homes : xx / 0.45-0.80
facilities for culture leisure and sports
entertainment culture : 60 / 1.67
large cinemas : 120 / 0.83
musical theatres : 70 / 1.43
large discos fun leisure pools : 60 / 0.8
arenas large events : 70 / 1.43
theme parks : 60 / 1.67
fitness wellness : 125 / 0.8
"""

def parse_weight(value_str):
    value_str = value_str.strip().replace(",", ".")
    
    if "-" in value_str:
        a, b = value_str.split("-", 1)
        return (float(a.strip()) + float(b.strip())) / 2
    else:
        return float(value_str)

lines = [line.strip() for line in text.splitlines() if line.strip()]

label_weights = {}
group_children = {}
current_group = None

for line in lines:
    # case 1: line has actual value
    if ":" in line and "/" in line:
        label, right_part = line.split(":", 1)
        label = label.strip()
        after_slash = right_part.split("/")[-1].strip()

        try:
            weight = parse_weight(after_slash)
            label_weights[label] = weight

            # if currently inside a main class, attach this subclass to it
            if current_group is not None:
                group_children.setdefault(current_group, []).append(weight)

        except ValueError:
            print(f"Could not parse: {line}")

    # case 2: line has no value => treat as new main class heading
    else:
        current_group = line
        group_children.setdefault(current_group, [])

# now compute average for group headings
for group, values in group_children.items():
    if values:
        label_weights[group] = sum(values) / len(values)

# round all values
label_weights = {k: round(v, 2) for k, v in label_weights.items()}

print(label_weights)

{'transport': 0.5, 'yards depots storage areas construction yards': 0.85, 'highly productive industries machine material or space intensive': 0.85, 'others industrial': 1.85, 'craft businesses': 1.9, 'craft courtyards': 1.85, 'normal Office': 2.9, 'open plan Office': 4.15, 'business oriented services': 3.5, 'customer oriented services': 3.0, 'Hotels': 1.5, 'hotels with conference Areas': 0.8, 'restaurants gastronomy': 1.9, 'suppliers for car dealerships': 1.7, 'vehicle electrical repair': 2.0, 'customer Service': 3.3, 'car dealerships': 0.7, 'wholesale': 2.45, 'retail small scale': 3.75, 'discount stores': 0.9, 'diy stores': 0.75, 'furniture stores': 0.55, 'hypermarkets superstores': 1.2, 'shopping centers': 3.1, 'self service department stores': 1.1, 'department stores': 1.55, 'factory outlet centers': 2.15, 'schools': 1.0, 'universities': 0.75, 'research institutes': 1.25, 'kindergartens': 2.3, 'hospitals': 1.25, 'nursing homes': 0.62, 'entertainment culture': 1.67, 'large cinemas': 

In [19]:
# Normalize weight dictionary keys to lowercase
weight_dict = {str(k).strip().lower(): v for k, v in label_weights.items()}

# Calculate potentials directly
df_final["potentials"] = (
    df_final["volume_m3"] *
    df_final["bosserhof_class"]
      .astype(str)
      .str.strip()
      .str.lower()
      .replace(["none", "nan"], np.nan)
      .map(weight_dict)
).round(2)

df_final.head()

,gml_id,volume_m3,osm_names,bosserhof_class,mid_label,geometry,potentials
0,13,89.209230,None,normal office,['work'],MULTIPOLYGON Z (((608781.076 5797034.372 86.80...,258.71
1,14,119.566130,None,normal office,['work'],"MULTIPOLYGON Z (((609122.06 5797462.983 85.46,...",346.74
2,15,1033.650498,None,normal office,['work'],MULTIPOLYGON Z (((609428.62 5797079.031 79.962...,2997.59
3,16,46.560366,None,normal office,['work'],"MULTIPOLYGON Z (((609661.688 5797071.969 77.4,...",135.03
4,0,931.423451,None,normal office,['work'],MULTIPOLYGON Z (((609554.181 5797264.172 78.94...,2701.13


In [20]:
df_final.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 263513 entries, 0 to 576280
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   gml_id           263513 non-null  object  
 1   volume_m3        263513 non-null  float64 
 2   osm_names        21393 non-null   object  
 3   bosserhof_class  263513 non-null  object  
 4   mid_label        263513 non-null  object  
 5   geometry         263513 non-null  geometry
 6   potentials       263512 non-null  float64 
dtypes: float64(2), geometry(1), object(4)
memory usage: 16.1+ MB


In [21]:
df_final = df_final.dropna(subset=['bosserhof_class'])
df_final.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 263513 entries, 0 to 576280
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   gml_id           263513 non-null  object  
 1   volume_m3        263513 non-null  float64 
 2   osm_names        21393 non-null   object  
 3   bosserhof_class  263513 non-null  object  
 4   mid_label        263513 non-null  object  
 5   geometry         263513 non-null  geometry
 6   potentials       263512 non-null  float64 
dtypes: float64(2), geometry(1), object(4)
memory usage: 16.1+ MB


In [22]:
df_final = df_final[df_final["volume_m3"] >= 30]

In [23]:
df_final.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 263513 entries, 0 to 576280
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   gml_id           263513 non-null  object  
 1   volume_m3        263513 non-null  float64 
 2   osm_names        21393 non-null   object  
 3   bosserhof_class  263513 non-null  object  
 4   mid_label        263513 non-null  object  
 5   geometry         263513 non-null  geometry
 6   potentials       263512 non-null  float64 
dtypes: float64(2), geometry(1), object(4)
memory usage: 16.1+ MB


In [24]:
df_final[df_final['potentials'].isna()]

,gml_id,volume_m3,osm_names,bosserhof_class,mid_label,geometry,potentials
536171,193231,39.823627,None,retail wholesale,['work'],MULTIPOLYGON Z (((621464.662 5778044.183 106.1...,NaN


In [25]:
df_final = df_final.dropna(subset=['potentials'])
df_final.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 263512 entries, 0 to 576280
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   gml_id           263512 non-null  object  
 1   volume_m3        263512 non-null  float64 
 2   osm_names        21393 non-null   object  
 3   bosserhof_class  263512 non-null  object  
 4   mid_label        263512 non-null  object  
 5   geometry         263512 non-null  geometry
 6   potentials       263512 non-null  float64 
dtypes: float64(2), geometry(1), object(4)
memory usage: 16.1+ MB


In [26]:
df_final.to_file("Areas-of-interest-POIs/final_potentials_calculation.gpkg", layer="final_potentials_calculation", driver="GPKG")